In [ ]:
# ============================================================
# 環境設定（Colab/ローカル共通）
# ============================================================
import os

# Google Earth Engineプロジェクト ID（ご自身のGEEプロジェクトIDに変更）
GEE_PROJECT = os.environ.get('GEE_PROJECT', 'your-ee-project-id')

# 出力ディレクトリ（Colabの場合は/content/drive/MyDrive/...、ローカルの場合は任意のパス）
OUTPUT_DIR = os.environ.get('OUTPUT_DIR', '/content/drive/MyDrive/Downscaling')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'GEE_PROJECT: {GEE_PROJECT}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')


In [1]:
!pip install rasterio
!pip install rioxarray
!pip install cftime
import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from shapely.geometry import box
from collections import defaultdict
import os
import cftime
from google.colab import files



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 63.7 MB/s eta 0:00:00


In [2]:
# ------------------------
# GCM Model Configuration
# ------------------------
# Uncomment the models you want to use.
# The 3 models below (Default) are the top scorers from 1_GCMsSelection.
# To use more models, uncomment the corresponding lines.

MODEL_CFG = {
    # === Default: Top 3 models from 1_GCMsSelection ===
    "ACCESS-CM2": ("r1i1p1f1", "gn"),
    "CanESM5": ("r1i1p1f1", "gn"),
    "EC-Earth3-Veg-LR": ("r1i1p1f1", "gr"),

    # === Other available models (uncomment to use) ===
    # "ACCESS-ESM1-5": ("r1i1p1f1", "gn"),
    # "BCC-CSM2-MR": ("r1i1p1f1", "gn"),
    # "CESM2": ("r4i1p1f1", "gn"),
    # "CESM2-WACCM": ("r3i1p1f1", "gn"),
    # "CMCC-CM2-SR5": ("r1i1p1f1", "gn"),
    # "CMCC-ESM2": ("r1i1p1f1", "gn"),
    # "CNRM-CM6-1": ("r1i1p1f2", "gr"),
    # "CNRM-ESM2-1": ("r1i1p1f2", "gr"),
    # "EC-Earth3": ("r1i1p1f1", "gr"),
    # "FGOALS-g3": ("r3i1p1f1", "gn"),
    # "GFDL-CM4_gr1": ("r1i1p1f1", "gr1"),
    # "GFDL-ESM4": ("r1i1p1f1", "gr1"),
    # "GISS-E2-1-G": ("r1i1p1f2", "gn"),
    # "HadGEM3-GC31-LL": ("r1i1p1f3", "gn"),
    # "HadGEM3-GC31-MM": ("r1i1p1f3", "gn"),
    # "IITM-ESM": ("r1i1p1f1", "gn"),
    # "INM-CM4-8": ("r1i1p1f1", "gr1"),
    # "INM-CM5-0": ("r1i1p1f1", "gr1"),
    # "IPSL-CM6A-LR": ("r1i1p1f1", "gr"),
    # "KACE-1-0-G": ("r1i1p1f1", "gr"),
    # "KIOST-ESM": ("r1i1p1f1", "gr1"),
    # "MIROC-ES2L": ("r1i1p1f2", "gn"),
    # "MIROC6": ("r1i1p1f1", "gn"),
    # "MPI-ESM1-2-HR": ("r1i1p1f1", "gn"),
    # "MPI-ESM1-2-LR": ("r1i1p1f1", "gn"),
    # "MRI-ESM2-0": ("r1i1p1f1", "gn"),
    # "NESM3": ("r1i1p1f1", "gn"),
    # "NorESM2-LM": ("r1i1p1f1", "gn"),
    # "NorESM2-MM": ("r1i1p1f1", "gn"),
    # "TaiESM1": ("r1i1p1f1", "gn"),
    # "UKESM1-0-LL": ("r1i1p1f2", "gn"),
}

models = list(MODEL_CFG.keys())
print(f"Number of models: {len(models)}")
for i, m in enumerate(models, 1):
    print(f"  {i}. {m}")

def base_model_name(model_key: str) -> str:
    """Convert pseudo keys back to real model name (e.g., GFDL-CM4_gr1 -> GFDL-CM4)."""
    return model_key.split("_")[0]


In [3]:
# ------------------------
# Study Area Polygon Input (geojson file)
# ------------------------
uploaded = files.upload()
geojson_file = list(uploaded.keys())[0]
cagayan = gpd.read_file(geojson_file).to_crs("EPSG:4326")

Saving test.geojson to test.geojson


In [4]:
# ------------------------
# GCM Mesh Points Creation（by 2000 ACCESS-CM2）crs="EPSG:4326"
# ------------------------
test_url = "https://nex-gddp-cmip6.s3-us-west-2.amazonaws.com/NEX-GDDP-CMIP6/ACCESS-CM2/historical/r1i1p1f1/pr/pr_day_ACCESS-CM2_historical_r1i1p1f1_gn_2000.nc"
ds_test = xr.open_dataset(test_url)
lon_raw = ds_test.lon.values
lat_raw = ds_test.lat.values

lon = np.atleast_1d(lon_raw)
lat = np.atleast_1d(lat_raw)
lon_grid, lat_grid = np.meshgrid(lon, lat)
res = 0.25  # GCM resolution
polygons = []
ids = []
for i, lat_val in enumerate(lat):
    for j, lon_val in enumerate(lon):
        cell = box(lon_val - res/2, lat_val - res/2, lon_val + res/2, lat_val + res/2)
        polygons.append(cell)
        ids.append(len(ids))

gdf_grid = gpd.GeoDataFrame({'id': ids}, geometry=polygons, crs="EPSG:4326")
gdf_grid["centroid"] = gdf_grid.geometry.centroid
cagayan_union = cagayan.union_all()
valid_points = gdf_grid[gdf_grid["centroid"].within(cagayan_union)].copy()
valid_points = valid_points.reset_index(drop=True)
id_list = list(range(len(valid_points)))
centroids = valid_points["centroid"]
lat_list = centroids.y.values
lon_list = centroids.x.values

print(f"✅ Number of Grids in the AOI: {len(valid_points)}")

/tmp/ipython-input-2985584064.py:22: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_grid["centroid"] = gdf_grid.geometry.centroid


✅ Number of Grids in the AOI: 3532


/tmp/ipython-input-2985584064.py:23: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  cagayan_union = cagayan.unary_union


In [5]:
# ------------------------
# Data Obtaining by year
# ------------------------
def fetch_model_data(model_key, scenario, years):
    model_data = defaultdict(list)

    # Read (member_id, grid_label) from the single dictionary
    member, grid = MODEL_CFG[model_key]

    # Resolve the real base model name used in S3 folder and filename
    base = base_model_name(model_key)

    for year in years:
        try:
            url = (
                "https://nex-gddp-cmip6.s3-us-west-2.amazonaws.com/NEX-GDDP-CMIP6/"
                f"{base}/{scenario}/{member}/pr/"
                f"pr_day_{base}_{scenario}_{member}_{grid}_{year}.nc"
            )

            # Open dataset and create YYYY/MM/DD strings
            try:
                ds = xr.open_dataset(url)
                ds["pr"] = ds["pr"] * 86400  # kg m-2 s-1 -> mm/day
                times = pd.to_datetime(ds.time.values).strftime("%Y/%m/%d")
            except Exception:
                ds = xr.open_dataset(url, decode_times=False)
                ds["pr"] = ds["pr"] * 86400
                times = [
                    str(cftime.num2date(t, ds.time.units, ds.time.calendar))[:10].replace("-", "/")
                    for t in ds.time.values
                ]

            # Sample nearest grid cell for each centroid point
            for idx, (lat, lon) in enumerate(zip(lat_list, lon_list)):
                vals = ds.sel(lat=lat, lon=lon, method="nearest")["pr"].values
                for d, v in zip(times, vals):
                    model_data[idx].append((d, float(v)))

        except Exception as e:
            print(f"❌ Error: {model_key} {scenario} {year} → {url} → {e}")

    return model_data


In [6]:
# ------------------------
# Historical Data Collection
# ------------------------
his_data = defaultdict(dict)
for model in models:
    print(f"\n🌍 Historical Processing: {model}")
    data = fetch_model_data(model, "historical", historical_years)
    his_data[model] = data

for idx in id_list:
    df = pd.DataFrame()
    for model in models:
        df_model = pd.DataFrame(his_data[model][idx], columns=["date", model])
        if df.empty:
            df = df_model
        else:
            df = df.merge(df_model, on="date", how="outer")
    df = df.sort_values("date")
    df.to_csv(f"{output_dir}/his_id_{idx}.csv", index=False)


🌍 Historical Processing: ACCESS-CM2


KeyboardInterrupt: 

In [7]:
# ------------------------
# SSP Scenario Download
# ------------------------
for ssp in ssp_scenarios:
    ssp_data = defaultdict(dict)
    print(f"\n📈 SSP Processing: {ssp}")
    for model in models:
        data = fetch_model_data(model, ssp, future_years)
        ssp_data[model] = data

    for idx in id_list:
        df = pd.DataFrame()
        for model in models:
            df_model = pd.DataFrame(ssp_data[model][idx], columns=["date", model])
            if df.empty:
                df = df_model
            else:
                df = df.merge(df_model, on="date", how="outer")
        df = df.sort_values("date")
        df.to_csv(f"{output_dir}/fut_{ssp}_id_{idx}.csv", index=False)

print("✅ All Model, Scenario are Downloaded")


📈 SSP Processing: ssp126


KeyboardInterrupt: 

In [8]:
import shutil
from google.colab import files

# ZIP file setting
zip_path = "/content/future_ssp_csvs.zip"

shutil.make_archive(base_name=zip_path.replace('.zip', ''), format='zip', root_dir=output_dir)

# Show download link
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
import shutil
from google.colab import files

# Output file
shp_dir = "/content/gcm_grid_shp"
os.makedirs(shp_dir, exist_ok=True)

# 1. Save Mesh Polygon（geometry = grid cell）
mesh_path = os.path.join(shp_dir, "gcm_grid_mesh.shp")
valid_points[["id", "geometry"]].to_file(mesh_path)

# 2. Save centroids（geometry = centroid）
centroid_path = os.path.join(shp_dir, "gcm_grid_centroid.shp")
valid_centroids = gpd.GeoDataFrame({"id": id_list}, geometry=valid_points.geometry.centroid, crs="EPSG:4326")
valid_centroids.to_file(centroid_path)

# Download with zip
shp_zip_path = "/content/gcm_grid_shapefile.zip"
shutil.make_archive(base_name=shp_zip_path.replace(".zip", ""), format="zip", root_dir=shp_dir)
files.download(shp_zip_path)


/tmp/ipython-input-1199283784.py:14: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  valid_centroids = gpd.GeoDataFrame({"id": id_list}, geometry=valid_points.geometry.centroid, crs="EPSG:4326")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')